In [1]:
import starfile
import dynamotable
import mrcfile
import pandas as pd
from pathlib import Path
from scipy.spatial.transform.rotation import Rotation as R

/tmp/ipykernel_61656/3596205805.py:6: DeprecationWarning: Please import `Rotation` from the `scipy.spatial.transform` namespace; the `scipy.spatial.transform.rotation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.spatial.transform.rotation import Rotation as R


In [2]:
tbl_file = "/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/dynamo_CJS_2025/alignment_projects/sirt_man550_best_workflow/top54_manual550_sirt_MaskOutAuNPsandMembrane_r6/results/ite_0001/averages/refined_table_ref_001_ite_0001.tbl"
tomo_doc = "/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/dynamo_CJS_2025/bin1_tomograms_sirt_top54.doc"

In [3]:
dynamo_data = dynamotable.read(tbl_file)

tomo_data = pd.read_csv(tomo_doc,delim_whitespace=True,header=None,names=['tomo','tomo_file'],dtype={'tomo':'int64'})
tomo_data['rlnTomoSizeX'] = 0
tomo_data['rlnTomoSizeY'] = 0
tomo_data['rlnTomoSizeZ'] = 0
for index, tomo_row in tomo_data.iterrows():
    tomo_path = Path("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS") / tomo_row['tomo_file']
    with mrcfile.open(tomo_path,mode='r',header_only=True) as mrc:
        tomo_data.at[index,'rlnTomoSizeX'] = mrc.header.nx
        tomo_data.at[index,'rlnTomoSizeY'] = mrc.header.ny
        tomo_data.at[index,'rlnTomoSizeZ'] = mrc.header.nz

print("Got tomo sizes")
combined = dynamo_data.merge(tomo_data,on='tomo',how='inner')
tomo_center_x = combined['rlnTomoSizeX'] / 2
tomo_center_y = combined['rlnTomoSizeY'] / 2
tomo_center_z = combined['rlnTomoSizeZ'] / 2
combined['rlnCenteredCoordinateXAngst'] = ((combined['x'] + combined['dx']) - tomo_center_x) * 2.5
combined['rlnCenteredCoordinateYAngst'] = ((combined['y'] + combined['dy']) - tomo_center_y) * 2.5
combined['rlnCenteredCoordinateZAngst'] = ((combined['z'] + combined['dz']) - tomo_center_z) * 2.5
combined['rlnAngleRot'] = combined['narot'] - 270
combined['rlnAngleTilt'] = - combined['tilt']
combined['rlnAnglePsi'] = combined['tdrot'] - 90
# Fix the angular range
eulers = R.from_euler(seq="ZYZ", angles=combined[["rlnAngleRot","rlnAngleTilt","rlnAnglePsi"]].values, degrees=True)
combined["rlnAngleRot"] = eulers.as_euler("ZYZ", degrees=True)[:, 0]
combined["rlnAngleTilt"] = eulers.as_euler("ZYZ", degrees=True)[:, 1]
combined["rlnAnglePsi"] = eulers.as_euler("ZYZ", degrees=True)[:, 2]
combined['rlnTomoName'] = combined['tomo_file'].str.split('/').str[0]

relion_df = combined[['rlnCenteredCoordinateXAngst','rlnCenteredCoordinateYAngst','rlnCenteredCoordinateZAngst','rlnAngleRot','rlnAngleTilt','rlnAnglePsi','rlnTomoName']]

print(relion_df)

starfile.write({"particles":relion_df}, f"/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/relion5_2025/from_dynamo/{Path(tbl_file).stem}.star")

/home/gouauxlab/andecath/env/lib/python3.11/site-packages/dynamotable/io.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(table_file, header=None, delim_whitespace=True)
/home/gouauxlab/andecath/env/lib/python3.11/site-packages/dynamotable/io.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipykernel_61656/724967733.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  tomo_data = pd.read_csv(tomo_doc,delim_whitespace=True,header=None,names=['tomo','tomo_file'],dtype={'tomo':'int64'})


Got tomo sizes
     rlnCenteredCoordinateXAngst  rlnCenteredCoordinateYAngst  \
0                     870.049275                  -330.880000   
1                    -144.768000                   724.100000   
2                     613.294000                  -419.733500   
3                     563.247500                  -496.781000   
4                    1762.303500                 -2382.402500   
..                           ...                          ...   
545                   167.989500                  1519.963250   
546                   -19.515000                   982.879375   
547                    10.901750                   824.308750   
548                 -4345.934250                  1650.481000   
549                  -251.840000                   -67.516250   

     rlnCenteredCoordinateZAngst  rlnAngleRot  rlnAngleTilt  rlnAnglePsi  \
0                    -469.410000     -76.9700        94.637     135.6280   
1                      84.467500    -115.4950       